# Sistema de búsqueda de estudiantes

Tienes un archivo con 10,000 estudiantes (nombre, edad,
promedio). Necesitas implementar un sistema que
permita:
1. Buscar un estudiante por su ID (número de matrícula)
2. Insertar nuevos estudiantes
3. Listar todos los estudiantes en orden por ID

Se desean organizar los datos por medio de: listas (nativas de python), árboles binarios tradicionales y árboles B+ para comparar sus desempeños sobre operaciones CRUD.

In [2]:
# Librerías relevantes
from faker import Faker
import random
import time
from statistics import mean
from abc import ABC, abstractmethod
import math

## Funciones y código relevante que será reutilizado durante todo el reto

In [3]:
class Estructura(ABC):
    """
        Clase abstracta, sirve para mantener la coherencia de las clases concretas de árbol binario, gestor de lista y árbol B+
        Todas deben mantener el contrato definido por los métodos de esta clase.
    """
    @abstractmethod
    def insertar(self, estudiante:dict) -> dict:
        pass

    @abstractmethod
    def buscar(self, id_estudiante:int) -> str:
        pass

    @abstractmethod
    def listar(self) -> str:
        pass

In [4]:
# Función generadora de estudiantes, en órden
def generar_estudiantes_orden(cantidad : int) -> list:
    faker = Faker() # Instanciamos faker para generar los datos
    estudiantes = []

    for i in range(1, cantidad + 1):
        estudiante = {"id": i, "nombre" : faker.name(), "promedio" : round(random.uniform(0, 10), 1)}
        estudiantes.append(estudiante)

    return estudiantes


# Función que genera estudiantes pero la lista no respeta el órden por id
def generar_estudiantes_sin_orden(cantidad : int) -> list:
    estudiantes = generar_estudiantes_orden(cantidad)
    random.shuffle(estudiantes)
    return estudiantes

# Función encargadd de medir el tiempo de búsqueda sobre una muestra de id's
def medir_tiempo_busqueda(estructura : Estructura, muestra):
    inicio = time.perf_counter()
    
    for id_objetivo in muestra:
        estructura.buscar(id_objetivo)

    fin = time.perf_counter()

    return fin - inicio

## Sistema de búsqueda, inserción y listado de todos los estudiantes

In [5]:
class GestorListaEstudiantes(Estructura):
    """
        Esta clase se encarga de gestionar las operaciones de inserción, búsqueda y listado de los estudiantes
        cuando la estructura de datos en cuestión es una lista nativa de Python.
    """
    def __init__(self, datos : list):
        self.datos = datos

    def insertar(self, estudiante:dict) -> dict:
        self.datos.append(estudiante)
        return estudiante

    def buscar(self, id_estudiante:int) -> str:
        for estudiante in self.datos:
            if estudiante["id"] == id_estudiante:
                resultado = ""
                resultado += "Encontrado :\n"
                for clave, valor in estudiante.items():
                    resultado += f"{clave} : {valor} | "
                return resultado

        return f"Estudiante con id: {id_estudiante} no fue encontrado."

    def listar(self) -> str:
        self.datos.sort(key= lambda estudiante : estudiante["id"])
        listado = ""
        for estudiante in self.datos:
            listado += "Estudiante:\n"
            listado += "["
            for clave, valor in estudiante.items():
                listado += f"{clave} : {valor} |"
            listado += "]\n"

        return listado      

In [6]:
class ABB(Estructura):
    """
        Esta clase representa una implementación básica de un árbol binario de búsqueda (ABB),
        hereda de la clase abstracta Estrucrura, para mantener los métodos de forma consistente con las
        otras estructuras de datos.

        Las operaciones de inserción, búsqueda y mostrar los datos son implementaciones basadas en los algoritmos
        del libro Introduction to Algorithms de Cormen.
    """

    class Nodo:
        """
            Esta clase representa un nodo de árbol binario de búsqueda.
            Para este reto, se decidió implementar el árbol a modo de lista ligada,
            es decir, el nodo guarda referencias hacia sus hijos.
        """
        def __init__(self, estudiante :dict, hijo_izquierdo : "ABB.Nodo" = None, hijo_derecho : "ABB.Nodo" = None):
            self.hijo_izquierdo = hijo_izquierdo
            self.hijo_derecho = hijo_derecho
            self.estudiante = estudiante

    def __init__(self):
        self.raiz = None #Raíz inicia siendo nula

    def insertar(self, estudiante :dict) -> dict:
        """
            Dado un diccionario que representa un estudiante, se inserta el nodo correspondiente en el árbol.
            Este es el algoritmo estándar (versión iterativa) de inserción en un árbol binario, adaptado del libro Introduction
            To Algorithms de Cormen.

            El procedimiento es el que sigue:
                1. nodo_actual se utiliza para recorrer el árbol y nodo_previo se utiliza para almacenar el nodo inmediatamente anterior
                    al nodo actual.
                2. Siempre que nodo_actual no sea nulo, desde él se decide si la inserción continúa por la izquierda (valor <=) o la derecha.
                3. Se actualiza el nodo_actual a su hijo izquierdo o derecho y se guarda en nodo_previo la referencia a él.
                4. Una vez que se llegue a una hoja, nodo_actual será nulo y nodo_previo será el padre del valor a insertar.
                5. Finalmente, se decide si el valor se inserta a la izquierda o a la derecha del nodo_previo.

        """
        nodo_actual = self.raiz
        nodo_previo = None
        while nodo_actual is not None:
            nodo_previo = nodo_actual
            if estudiante["id"] <= nodo_actual.estudiante["id"]:
                nodo_actual = nodo_actual.hijo_izquierdo
            else:
                nodo_actual = nodo_actual.hijo_derecho

        if nodo_previo is None:
            self.raiz = self.Nodo(estudiante=estudiante) # Caso especial: árbol totalmente vacío.
        elif estudiante["id"] <= nodo_previo.estudiante["id"]:
            nodo_previo.hijo_izquierdo = self.Nodo(estudiante=estudiante)
        else:
            nodo_previo.hijo_derecho = self.Nodo(estudiante=estudiante)
        
        return estudiante

    def buscar(self, id_estudiante : int) -> str:
        """
            Dado un id de estudiante, recorre el árbol para encontrar el nodo correspondiente.
            Este es el algoritmo estándar (versión iterativa) de búsqueda en un árbol binario, adaptado del libro
            Introduction To Algorithms de Cormen.

            El procedimiento es el que sigue:
                1. nodo_actual se utiliza para recorrer el árbol, comenzando desde la raíz.
                2. En cada iteración, si el id buscado coincide con el del nodo_actual, el ciclo termina (encontrado).
                3. Si no coincide, se decide si continuar por la izquierda (id buscado <=) o por la derecha,
                    aprovechando la propiedad de orden del ABB para descartar la mitad del subárbol en cada paso.
                4. El ciclo también termina si nodo_actual llega a ser nulo, lo que significa que el id
                    no existe en el árbol.
                5. Finalmente, se construye el string de resultado según si el nodo fue encontrado o no.

        """
        nodo_actual = self.raiz
        while nodo_actual is not None and id_estudiante != nodo_actual.estudiante["id"]:
            if id_estudiante <= nodo_actual.estudiante["id"]:
                nodo_actual = nodo_actual.hijo_izquierdo
            else:
                nodo_actual = nodo_actual.hijo_derecho

        resultado = ""
        if nodo_actual is not None:
            resultado += "Encontrado :\n"
            for clave, valor in nodo_actual.estudiante.items():
                resultado += f"{clave} : {valor} | "
        else:
            resultado = f"Estudiante con id: {id_estudiante} no fue encontrado."
    
        return resultado

    def listar(self) -> str:
        """
            Retorna un string con todos los estudiantes del árbol, ordenados de forma ascendente por id.

            Este método es un simple punto de entrada que delega el trabajo real en _inorden, el cual
            realiza un recorrido in-order (izquierda, nodo, derecha) sobre el árbol. Gracias a la propiedad
            de orden del ABB, este tipo de recorrido garantiza que los estudiantes se visiten en orden
            ascendente por id, sin necesidad de un paso adicional de ordenamiento (a diferencia de la
            lista, donde listar en orden requiere un sort completo).

            Se usa una lista auxiliar mutable (resultado) para ir acumulando los fragmentos de texto y luego
            unirlos con "".join(), en vez de concatenar strings directamente en cada llamada recursiva,
            ya que la concatenación repetida de strings es más costosa en tiempo (cada concatenación crea
            un nuevo string en memoria).
        """
        resultado = []
        self._inorden(self.raiz, resultado)
        return "".join(resultado)

    def _inorden(self, nodo : Nodo, resultado : list):
        """
            Recorrido in-order recursivo sobre el árbol, usado como método auxiliar de listar().

            El recorrido in-order visita primero el subárbol izquierdo, luego el nodo actual, y finalmente
            el subárbol derecho. Por la propiedad de orden de un ABB (todo lo que está a la izquierda de
            un nodo es menor, y todo lo que está a la derecha es mayor), este orden de visita produce
            los ids en secuencia ascendente.

            Caso base: si nodo es None, no hay nada que recorrer ni agregar, y la recursión termina
            en esa rama.
        """
        if nodo is None:
            return

        self._inorden(nodo.hijo_izquierdo, resultado)

        resultado.append("Estudiante:\n")
        datos = "["
        for clave, valor in nodo.estudiante.items():
            datos += f"{clave} : {valor} | "
        datos += "]\n"

        resultado.append(datos)

        self._inorden(nodo.hijo_derecho, resultado)

    def construir_arbol(self, datos : list) -> None:
        """
            Construye el árbol completo a partir de una lista de estudiantes, insertándolos uno por uno
            en el orden en que aparecen en la lista.
        """
        for estudiante in datos:
            self.insertar(estudiante)

In [7]:
class ArbolBPlus(Estructura):
    class Nodo:
        """
            Nodo del árbol B+ de orden n (n es el número total de punteros del nodo)
            Se sigue la estructura conceptual de este tipo de nodos:

            nodo = [Puntero 1, Clave 1, Puntero 2, Clave 2, ..., Puntero n-1, Clave n-1, Puntero n]

            Donde Puntero n representa al puntero que enlaza las hojas como lista ligada.
        """
        def __init__(self, orden_n : int, es_hoja : bool = True):
            self.orden_n = orden_n # Guardamos cúal es el órden n (número total de punteros que puede tener el nodo)
            self.es_hoja = es_hoja # Bandera booleana que diferencia a un nodo hoja de un nodo interno

            # A continuación, la estructura conceptual del nodo será representada mediante dos listas paralelas
            # Una lista guarda punteros, la otra guarda las claves.

            self.claves = [] # Guardamos las claves que van desde 1 hasta n-1
            self.punteros = [] # Guardamos solo los punteros que llevan hacia nodos hijos/datos

            self.puntero_siguiente = None # Atributo reservado para las HOJAS, apunta hacia la siguiente hoja de la lista enlazada.

            self.padre = None # Atributo útil para los algoritmos de inserción

        def tiene_espacio(self) -> bool:
            """
                Método interno diseñado para verificar si un nodo tiene espacio disponible o no.
                Si el nodo es hoja, validamos si su cantidad de claves es menor que n - 1.
                Si el nodo es interno, validamos si su cantidad de punteros es menor que n.
            """

            if self.es_hoja:
                return len(self.claves) < self.orden_n - 1
            else:
                return len(self.punteros) < self.orden_n



    def __init__(self, orden_n):
        self.raiz : "ArbolBPlus.Nodo" = None
        self.orden_n : int = orden_n


    def insertar(self, estudiante):
        clave_busqueda = estudiante["id"]
        puntero_dato = estudiante

        if self.raiz == None:
            nuevo_nodo = self.Nodo(orden_n=self.orden_n)
            self._insertar_en_hoja(nuevo_nodo, clave_busqueda, puntero_dato)
            self.raiz = nuevo_nodo
        else:
            hoja_objetivo = self._buscar_hoja(clave_busqueda)

            if hoja_objetivo.tiene_espacio():
                self._insertar_en_hoja(hoja_objetivo, clave_busqueda, puntero_dato)
            else:
                nodo_division = self.Nodo(orden_n=self.orden_n)

                punteros_hoja_objetivo = hoja_objetivo.punteros.copy()
                claves_hoja_objetivo = hoja_objetivo.claves.copy()

                nodo_temp = self.Nodo(self.orden_n)
                nodo_temp.claves = claves_hoja_objetivo
                nodo_temp.punteros = punteros_hoja_objetivo

                self._insertar_en_hoja(nodo_temp, clave_busqueda, puntero_dato)

                nodo_division.puntero_siguiente = hoja_objetivo.puntero_siguiente
                hoja_objetivo.puntero_siguiente = nodo_division

                limite = math.ceil(self.orden_n/2)

                hoja_objetivo.punteros = nodo_temp.punteros[:limite]
                hoja_objetivo.claves = nodo_temp.claves[:limite]

                nodo_division.punteros = nodo_temp.punteros[limite:]
                nodo_division.claves = nodo_temp.claves[limite:]

                clave_intermedia = nodo_division.claves[0]

                self._insertar_en_padre(hoja_objetivo, clave_intermedia, nodo_division)

    def buscar(self, id_estudiante: int) -> str:
        """
            Busca un estudiante por su id, descendiendo hasta la hoja correspondiente
            y revisando si la clave existe ahí.
        """
        hoja = self._buscar_hoja(id_estudiante)

        if hoja is not None:
            for i, clave in enumerate(hoja.claves):
                if clave == id_estudiante:
                    resultado = "Encontrado :\n"
                    for k, v in hoja.punteros[i].items():
                        resultado += f"{k} : {v} | "
                    return resultado

        return f"Estudiante con id: {id_estudiante} no fue encontrado."

    def listar(self) -> str:
        """
            Recorre todas las hojas del árbol de izquierda a derecha, aprovechando
            el enlace puntero_siguiente entre ellas, y devuelve los estudiantes
            en orden ascendente por id.
        """
        resultado = []
        hoja = self.raiz

        # Bajamos hasta la hoja más a la izquierda del árbol
        while hoja is not None and not hoja.es_hoja:
            hoja = hoja.punteros[0]

        # Recorremos las hojas enlazadas hacia adelante
        while hoja is not None:
            for estudiante in hoja.punteros:
                resultado.append("Estudiante:\n[")
                for k, v in estudiante.items():
                    resultado.append(f"{k} : {v} | ")
                resultado.append("]\n")
            hoja = hoja.puntero_siguiente

        return "".join(resultado)

    def construir_arbol(self, datos: list) -> None:
        """
            Inserta, uno por uno y en el orden dado, todos los estudiantes de la lista.
        """
        for estudiante in datos:
            self.insertar(estudiante)

            
    ## Métodos base (algoritmos de búsqueda e inserción), obtenidos de Database System Concepts (Silberschatz, Korth, Sudarshan)

    def _buscar_hoja(self, clave_busqueda : int):
        """
            Algoritmo estándar de búsqueda en un árbol B+.
            Desciende desde la raíz hasta encontrar la hoja que
            se espera contenga la clave de búsqueda indicada por el 
            parámetro clave_busqueda.
        """
        if self.raiz is None:
            return None
        
        nodo_actual = self.raiz

        while not nodo_actual.es_hoja:
            # Encontramos el índice i más pequeño tal que clave_busqueda <= nodo_actual.claves[i]
            i = next((i for i, clave in enumerate(nodo_actual.claves) if clave_busqueda <= clave), None)

            if i is None:
                # Si la clave objetivo es mayor a todas las de este nodo, descendemos hacia el hijo más grande.
                nodo_actual = nodo_actual.punteros[-1]

            elif clave_busqueda == nodo_actual.claves[i]:
                # Si la clave objetivo coincide con una clave en este nodo, entonces descendemos hacia la derecha, ya que
                # los valores mayores o iguales van al subárbol derecho.
                nodo_actual = nodo_actual.punteros[i + 1]
            else:
                # Si la clave objetivo es menor a la clave encontrada en i, descendemos por su puntero correspondiente.
                nodo_actual = nodo_actual.punteros[i]

        # Tras finalizar el ciclo, nos encontramos en la hoja donde se espera que se encuentre el valor
        return nodo_actual

    def _insertar_en_hoja(self, hoja : "ArbolBPlus.Nodo", clave : int, puntero : dict) -> None:

        # La hoja está completamente vacía
        if not hoja.claves:
            hoja.claves.append(clave)
            hoja.punteros.append(puntero)
            return

        # Buscamos la primera clave tal que sea mayor a la clave que se desea insertar
        for i in range(len(hoja.claves)):
            if hoja.claves[i] > clave:
                hoja.claves.insert(i, clave)
                hoja.punteros.insert(i, puntero)
                return

        # Si la clave es mayor que todas las demás, la insertamos al final
        hoja.claves.append(clave)
        hoja.punteros.append(puntero)
            

    def _insertar_en_padre(self, nodo_original : "ArbolBPlus.Nodo", clave_separadora : int, nodo_nuevo : "ArbolBPlus.Nodo") -> None:
        if nodo_original == self.raiz:
            nueva_raiz = self.Nodo(nodo_original.orden_n, es_hoja=False)
            nueva_raiz.claves = [clave_separadora]
            nueva_raiz.punteros = [nodo_original, nodo_nuevo]
            nodo_original.padre = nueva_raiz
            nodo_nuevo.padre = nueva_raiz
            self.raiz = nueva_raiz
            return

        padre = nodo_original.padre

        if padre.tiene_espacio():
            for i in range(len(padre.claves)):
                if padre.claves[i] > clave_separadora:
                    padre.claves.insert(i, clave_separadora)
                    padre.punteros.insert(i + 1, nodo_nuevo)
                    break
            else:
                # Si la clave resulta ser la mayor existente y el for nunca se detuvo
                padre.claves.append(clave_separadora)
                padre.punteros.append(nodo_nuevo)

            # Como ahora hay nuevos punteros en este padre
            nodo_nuevo.padre = padre
        else:
            # Se extraen copias de los punteros y claves del padre
            punteros_padre = padre.punteros.copy()
            claves_padre = padre.claves.copy()

            # Insertamos clave_separadora y el nuevo nodo en las copias
            for i in range(len(claves_padre)):
                if claves_padre[i] > clave_separadora:
                    claves_padre.insert(i, clave_separadora)
                    punteros_padre.insert(i + 1, nodo_nuevo)
                    break
            else:
                # Si la clave resulta ser la mayor existente y el for nunca se detuvo
                claves_padre.append(clave_separadora)
                punteros_padre.append(nodo_nuevo)

            # Ahora, procedemos a dividir al padre, creando un nuevo nodo
            nodo_division_padre = self.Nodo(orden_n=padre.orden_n, es_hoja=False)

            limite = math.ceil((nodo_division_padre.orden_n + 1) / 2)

            padre.punteros = punteros_padre[:limite]
            padre.claves = claves_padre[:limite - 1]

            nueva_clave_intermedia = claves_padre[limite - 1]

            nodo_division_padre.punteros = punteros_padre[limite:]
            nodo_division_padre.claves = claves_padre[limite:]

            # Los hijos que quedaron en el nuevo padre de la división deben actualizar sus punteros
            for nodo_hijo in nodo_division_padre.punteros:
                nodo_hijo.padre = nodo_division_padre

            # En caso de que el nodo nuevo se encuentre en el nodo padre original, se actualiza su puntero padre
            if nodo_nuevo in padre.punteros:
                nodo_nuevo.padre = padre
            else:
                nodo_nuevo.padre = nodo_division_padre

            self._insertar_en_padre(padre, nueva_clave_intermedia, nodo_division_padre)



### Pruebas y mediciones

In [8]:
# Prueba 1: datos ordenados por id en una lista

# Generamos la lista de estudiante, ordenada por id de forma creciente
lista_ordenada = generar_estudiantes_orden(cantidad= 10000)

# Instanciamos el gestor de la lista
gestor = GestorListaEstudiantes(datos=lista_ordenada)

# Realizamos 200 mediciones para hallar el promedio de tiempo que tarda el sistema en buscar 100 id's 
# escogidos aleatoriamente, cada intento se hace sobre una muestra de id's diferentes.
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(gestor, random.sample(range(1, len(lista_ordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista ordenada por id) es: {mean(resultados)}")


# Prueba 2: datos desordenados en una lista
lista_desordenada = generar_estudiantes_sin_orden(cantidad=10000)

# Lo siguiente es igual al código previo:
gestor = GestorListaEstudiantes(datos=lista_desordenada)
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(gestor, random.sample(range(1, len(lista_desordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista desordenada) es: {mean(resultados)}")

El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista ordenada por id) es: 0.09965420999993511
El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (lista desordenada) es: 0.13886368850016878


In [9]:
# Prueba 3: datos ordenados por id en un árbol binario

# Generamos la lista de estudiante, ordenada por id de forma creciente
lista_ordenada = generar_estudiantes_orden(cantidad= 10000)

# Instanciamos el árbol binario
arbol_abb = ABB()
arbol_abb.construir_arbol(datos=lista_ordenada)

# Realizamos 200 mediciones para hallar el promedio de tiempo que tarda el sistema en buscar 100 id's 
# escogidos aleatoriamente, cada intento se hace sobre una muestra de id's diferentes.
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_abb, random.sample(range(1, len(lista_ordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista ordenada por id) es: {mean(resultados)}")


# Prueba 4: datos desordenados en una lista dentro de un árbol binario
lista_desordenada = generar_estudiantes_sin_orden(cantidad=10000)

# Lo siguiente es igual al código previo:
arbol_abb = ABB()
arbol_abb.construir_arbol(datos=lista_desordenada)
resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_abb, random.sample(range(1, len(lista_desordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista desordenada) es: {mean(resultados)}")

El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista ordenada por id) es: 0.2216553284999827
El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol binario con lista desordenada) es: 0.0015845455004091492


In [10]:
# Prueba 5: datos ordenados por id en un árbol B+

# Generamos la lista de estudiantes, ordenada por id de forma creciente
lista_ordenada = generar_estudiantes_orden(cantidad=10000)

# Instanciamos el árbol B+ y lo construimos con los datos ordenados
arbol_bplus = ArbolBPlus(orden_n=4)  
arbol_bplus.construir_arbol(datos=lista_ordenada)

resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_bplus, random.sample(range(1, len(lista_ordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol B+ con lista ordenada por id) es: {mean(resultados)}")


# Prueba 6: datos desordenados en un árbol B+
lista_desordenada = generar_estudiantes_sin_orden(cantidad=10000)

arbol_bplus = ArbolBPlus(orden_n=4)
arbol_bplus.construir_arbol(datos=lista_desordenada)

resultados = []
for _ in range(200):
    tiempo = medir_tiempo_busqueda(arbol_bplus, random.sample(range(1, len(lista_desordenada) + 1), 100))
    resultados.append(tiempo)

print(f"El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol B+ con lista desordenada) es: {mean(resultados)}")

El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol B+ con lista ordenada por id) es: 0.002786916499644576
El promedio de tiempo (s) de búsqueda de 100 estudiantes por id (árbol B+ con lista desordenada) es: 0.0027134655001282227
